In [1]:
!pip install clustering-benchmarks -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 47.1 MB/s eta 0:00:00


In [2]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [3]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

| Battery | Dataset    | Rows   | Cols   | Clusters
|---------|------------|--------|--------|----------
| uci     | ecoli      | 336    | 7      | 8
| uci     | glass      | 214    | 9      | 6
| uci     | ionosphere | 351    | 33     | 2
| uci     | sonar      | 208    | 60     | 2
| uci     | statlog    | 2310   | 18     | 7
| uci     | wdbc       | 569    | 30     | 2
| uci     | wine       | 178    | 13     | 3
| uci     | yeast      | 1484   | 8      | 4
| mnist   | digits     | 70000  | 719    | 10
| mnist   | fashion    | 70000  | 784    | 10
| sipu    | worms_64   | 105000 | 64     | 25


In [4]:
import numpy as np
from sklearn.neural_network import MLPRegressor
import pandas as pd

def ssnn_feat_eng(X, n_embeddings=2):

  # To store embeddings for each feature
  embeddings_list = []

  # Get the number of features
  n_samples, n_features = X.shape

  for feature_idx in range(n_features):

      # Prepare input by removing the current feature
      X_in = np.delete(X, feature_idx, axis=1)  # shape: (n_samples, n_features-1)

      # Target is the left-out feature
      y_target = X[:, feature_idx]  # shape: (n_samples,)

      # Define MLP
      mlp = MLPRegressor(
          hidden_layer_sizes=(64, n_embeddings),
          # Use 'identity' for linear activation in the hidden layers
          activation='logistic', #{'relu', 'logistic', 'identity', 'tanh'}
          max_iter=10000,
          random_state=42
      )

      # Fit the model
      mlp.fit(X_in, y_target)

      # Extract learned weights and biases
      coefs = mlp.coefs_
      intercepts = mlp.intercepts_

      # Forward pass to hidden layer 1 (linear)
      Z1 = X_in @ coefs[0] + intercepts[0]  # shape: (n_samples, 64)

      # Forward pass to hidden layer 2 (linear) → this is the embedding
      embeddings = Z1 @ coefs[1] + intercepts[1]  # shape: (n_samples, n_embeddings)

      # Append embeddings
      embeddings_list.append(embeddings)

  # Concatenate all embeddings horizontally
  X_new = np.hstack(embeddings_list)  # shape: (n_samples, n_embeddings * n_features)
  #X_new_df = pd.DataFrame(X_new, columns=[f'feature_{i+1}_emb_{j+1}' for i in range(n_features) for j in range(n_embeddings)])
  return X_new

In [5]:
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def autoencoder_feat_eng(X, n_embeddings=8):

  X = StandardScaler().fit_transform(X)
  X = torch.tensor(X, dtype=torch.float)

  # --- Autoencoder definition ---
  class Autoencoder(nn.Module):
      def __init__(self, input_dim, latent_dim=n_embeddings):
          super().__init__()
          self.encoder = nn.Sequential(
              nn.Linear(input_dim, 64),
              nn.ReLU(),
              nn.Linear(64, latent_dim)
          )
          self.decoder = nn.Sequential(
              nn.Linear(latent_dim, 64),
              nn.ReLU(),
              nn.Linear(64, input_dim)
          )

      def forward(self, x):
          z = self.encoder(x)
          x_hat = self.decoder(z)
          return x_hat, z

  # --- Train AE ---
  input_dim = X.shape[1] # Dynamically set input_dim
  model = Autoencoder(input_dim)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.MSELoss()

  for epoch in range(100):
      x_hat, z = model(X)
      loss = criterion(x_hat, X)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # --- Cluster latent space ---
  with torch.no_grad():
      latent = model.encoder(X).numpy()

  return latent

In [6]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [7]:
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765321,1.035127,0.001835,-0.000089,-0.020113,-0.097887,-0.024911,-0.146013,0.428179,0.372140,0.588548,0.323847,-0.168114,0.481106,-0.312991,0.570539,-0.002047,-0.012851
1,-0.225939,0.124837,-0.000273,-0.000089,-0.030648,-0.103515,-0.039662,-0.149412,-0.685803,-0.622436,-0.789531,-0.645442,0.190102,-0.311183,0.121081,-0.807541,0.010869,-0.014419
2,1.461891,-1.562994,-0.000272,-0.000090,-0.018006,-0.093629,-0.024911,-0.136887,1.630661,1.499469,1.812804,1.579712,-0.393579,0.546427,-0.152849,1.794796,-0.004316,-0.017770
3,-1.762054,0.940306,-0.000273,-0.000089,-0.003257,-0.074487,0.124696,-0.028337,0.124047,0.127711,0.165009,0.079418,0.010996,0.122889,-0.133884,0.147002,-0.003034,-0.012061
4,-1.212087,1.395451,-0.000273,-0.000089,-0.008525,-0.079536,0.003536,-0.119822,0.237832,0.216212,0.329368,0.167919,-0.064863,0.274606,-0.209742,0.311358,-0.002351,-0.012504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799984,-0.406166,-0.000272,-0.000089,-0.012737,-0.106026,-0.020698,-0.141165,-0.318457,-0.236827,-0.363885,-0.354655,0.244890,-0.136290,-0.108600,-0.381894,-0.000868,-0.003645
2306,0.342992,-1.885389,-0.000272,-0.000090,-0.011685,-0.091065,-0.029126,-0.134704,1.717758,1.609040,1.848626,1.695606,-0.326150,0.392606,-0.066458,1.830617,-0.004883,-0.018707
2307,-0.851764,-0.975097,-0.000272,-0.000089,-0.012738,-0.089239,-0.018591,-0.134197,0.416239,0.351068,0.573797,0.323848,-0.195507,0.472676,-0.277170,0.555788,-0.002130,-0.013792
2308,-0.510404,0.181730,-0.000272,-0.000090,-0.025383,-0.105009,-0.038608,-0.150120,-0.684399,-0.622436,-0.785317,-0.645443,0.185888,-0.302756,0.116868,-0.803325,0.010868,-0.014420


In [8]:
ssnn_embeddings = ssnn_feat_eng(b.data)

In [9]:
autoencoder_embeddings = autoencoder_feat_eng(b.data)

In [10]:
ssnn_plus_autoencoder_embeddings = np.hstack((ssnn_embeddings, autoencoder_embeddings))

In [13]:
pd.DataFrame(ssnn_plus_autoencoder_embeddings)

,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,40,41,42,43
0,13.810085,-14.820672,-1.390467,1.381657,0.542239,-0.713926,0.543542,-0.715491,0.703280,-0.889853,...,0.476079,-0.507681,-1.374573,-0.545369,1.294851,-0.077561,-0.142180,-0.406945,-0.143144,0.497645
1,-18.471587,18.313972,39.594624,-29.800385,0.719512,-0.619814,0.717585,-0.617696,0.534081,-0.376234,...,1.052364,-1.093461,1.537905,1.573293,-1.396485,-1.106221,-0.636895,0.048001,-0.097723,-0.674662
2,66.275983,-69.370006,-54.023980,41.213041,0.370178,-0.534254,0.371964,-0.536160,0.202700,-0.304570,...,-0.424123,0.776374,-1.944125,-2.461271,2.343422,1.296672,-0.056913,-0.070926,1.905120,1.028718
3,-2.503999,2.719886,3.023179,-2.085312,0.576235,-0.656969,0.574948,-0.655326,0.799174,-0.930175,...,0.705101,-0.736405,-0.108335,-0.223338,0.535418,-0.158386,0.121930,0.377812,0.159840,0.108027
4,7.451306,-8.738280,-3.546143,2.966612,0.561520,-0.706983,0.559853,-0.704888,0.790987,-0.969680,...,0.719297,-0.808102,-0.438608,-0.487646,0.614006,0.262016,0.269545,-0.168387,0.427778,-0.140450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-19.439699,21.569566,20.641672,-15.382502,0.717661,-0.646674,0.715344,-0.643927,0.585201,-0.466575,...,0.569663,-0.459580,0.499998,0.501026,-0.574032,-0.294268,-0.287336,-0.492338,-0.002490,-0.646560
2306,70.514259,-73.997600,-61.007842,46.475206,0.354626,-0.497120,0.354999,-0.497248,0.103546,-0.162835,...,-0.411241,0.794745,-1.834734,-2.569873,2.259203,1.259596,-0.052673,-0.202002,2.152013,1.007870
2307,11.728603,-10.590088,-11.766128,9.232393,0.624280,-0.648388,0.624903,-0.649072,0.647771,-0.661657,...,-0.172852,0.454189,-0.918042,-0.898974,1.277328,0.660130,-0.084420,-0.678726,0.758432,0.125623
2308,-18.605382,18.438070,38.463058,-28.941711,0.719783,-0.624501,0.717638,-0.622106,0.551861,-0.401829,...,1.054666,-1.098894,1.536479,1.594914,-1.400078,-1.099030,-0.651673,0.048635,0.015006,-0.690325


In [14]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = b.data
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [15]:
import genieclust
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_ssnn_plus_autoencoder(battery, dataset, apply_scale=False, ssnn_embeddings=2, autoencoder_embeddings=2):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  ssnn_embeddings = ssnn_feat_eng(b.data, n_embeddings=ssnn_embeddings)
  autoencoder_embeddings = autoencoder_feat_eng(b.data, n_embeddings=autoencoder_embeddings)
  X_transformed = np.hstack((b.data, ssnn_embeddings, autoencoder_embeddings))

  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [16]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Datasets"):
  for dataset in battery_datasets_dict[battery]:
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 2/2 [00:26<00:00, 13.36s/it]


In [17]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,1.000000
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,1.000000
7,fcps,twodiamonds,0.992500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.435664


In [18]:
ssnn_embeddings_list = [2,4,8,16,32]
autoencoder_embeddings_list = [2,4,16,32,64,128]

scores_lists = {}
for ssnn_embs in tqdm.tqdm(ssnn_embeddings_list, desc="Processing Datasets"):
  for autoencoder_embs in autoencoder_embeddings_list:
    for battery in battery_datasets_dict.keys():
      for dataset in battery_datasets_dict[battery]:
        column_name = f'Genie+SSNN{ssnn_embs})+AE{autoencoder_embs} NCA Score'
        if column_name not in scores_lists:
          scores_lists[column_name] = []
        scores_lists[column_name].append(get_scores_with_ssnn_plus_autoencoder(battery, dataset,
                                                                             ssnn_embeddings=ssnn_embs,
                                                                             autoencoder_embeddings=autoencoder_embs))

df_ssnn_plus_autoencoder = pd.DataFrame.from_dict(scores_lists)

Processing Datasets: 100%|██████████| 5/5 [1:20:30<00:00, 966.03s/it]


In [19]:
df = pd.concat([df, df_ssnn_plus_autoencoder], axis=1)
numerical_cols = df.columns[2:]
df.style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+SSNN2)+AE2 NCA Score,Genie+SSNN2)+AE4 NCA Score,Genie+SSNN2)+AE16 NCA Score,Genie+SSNN2)+AE32 NCA Score,Genie+SSNN2)+AE64 NCA Score,Genie+SSNN2)+AE128 NCA Score,Genie+SSNN4)+AE2 NCA Score,Genie+SSNN4)+AE4 NCA Score,Genie+SSNN4)+AE16 NCA Score,Genie+SSNN4)+AE32 NCA Score,Genie+SSNN4)+AE64 NCA Score,Genie+SSNN4)+AE128 NCA Score,Genie+SSNN8)+AE2 NCA Score,Genie+SSNN8)+AE4 NCA Score,Genie+SSNN8)+AE16 NCA Score,Genie+SSNN8)+AE32 NCA Score,Genie+SSNN8)+AE64 NCA Score,Genie+SSNN8)+AE128 NCA Score,Genie+SSNN16)+AE2 NCA Score,Genie+SSNN16)+AE4 NCA Score,Genie+SSNN16)+AE16 NCA Score,Genie+SSNN16)+AE32 NCA Score,Genie+SSNN16)+AE64 NCA Score,Genie+SSNN16)+AE128 NCA Score,Genie+SSNN32)+AE2 NCA Score,Genie+SSNN32)+AE4 NCA Score,Genie+SSNN32)+AE16 NCA Score,Genie+SSNN32)+AE32 NCA Score,Genie+SSNN32)+AE64 NCA Score,Genie+SSNN32)+AE128 NCA Score
0,fcps,atom,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,fcps,chainlink,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,fcps,engytime,0.918870,0.918870,0.918870,0.918870,0.918870,0.918870,0.918870,0.942337,0.916915,0.953579,0.916426,0.918381,0.926202,0.918870,0.918870,0.918870,0.918870,0.918870,0.918870,0.755348,0.755348,0.755348,0.755348,0.755348,0.755348,0.690478,0.690967,0.694390,0.694390,0.694390,0.695366
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,fcps,tetra,1.000000,1.000000,1.000000,0.996667,1.000000,0.583333,0.996667,1.000000,1.000000,1.000000,0.996667,0.996667,1.000000,1.000000,0.996667,1.000000,0.996667,1.000000,0.996667,1.000000,1.000000,0.996667,0.996667,0.566667,0.996667,1.000000,1.000000,1.000000,1.000000,0.996667,0.996667
7,fcps,twodiamonds,0.992500,0.532500,0.990000,0.990000,0.990000,0.992500,0.990000,0.990000,0.880000,0.990000,0.990000,0.990000,0.990000,0.987500,0.532500,0.990000,0.990000,0.990000,0.990000,0.857500,0.992500,0.990000,0.990000,0.990000,0.882500,0.990000,0.992500,0.990000,0.990000,0.990000,0.990000
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,uci,ecoli,0.435664,0.405951,0.409291,0.409291,0.405951,0.405951,0.409291,0.392308,0.392308,0.392308,0.392308,0.392308,0.510618,0.375125,0.375125,0.367240,0.375125,0.367240,0.374661,0.300200,0.300200,0.300200,0.300200,0.300200,0.300200,0.336642,0.318474,0.380641,0.425767,0.453382,0.347988
